# Mean-variance optimizer (MVT) for FantasyGo

The "beat the opponent" optimizer of FPL-Optimal-Transfer (`FPL Data/beta estimators/opponent betas/optimization.ipynb`),
applied to the FantasyGo contests of 2023-24.

Choose the team $w$ that maximizes the probability of beating a benchmark score:

$$\max_{w\in\mathbb{W}} \text{Prob}\{w^T\delta > G^{(r')}(\mathbb{W}_o, \delta)\}$$

* $\delta$: the uncertain points of every player, $\delta \sim N(\mu_\delta, \Sigma_\delta)$
* $\mathbb{W}_o$: the opponents' teams; $G_o = w_o^T\delta$ is one team's score and $G^{(r')}$ the score at rank
  `R_PCT` among them (0.5 = beat the median team, 0.8 = finish in the top 20%)
* $\mathbb{W}$: the teams allowed by the FantasyGo rules

**Theorem** (mean-variance reduction): with $Y_w = w^T\delta - G^{(r')}$,
$\mu_{Y_w} = w^T\mu_\delta - \mu_{G}$ and $\sigma^2_{Y_w} = w^T\Sigma_\delta w + \sigma^2_{G} - 2w^T\sigma_{\delta,G}$, and the
solution is $w(\lambda) \in \arg\max_w \; w^T\mu_\delta \mp \lambda\,(w^T\Sigma_\delta w - 2w^T\sigma_{\delta,G})$
($-$ when $\mu_{Y_w}\geq 0$) for some $\lambda \geq 0$. $\mu_G$, $\sigma^2_G$ and $\sigma_{\delta,G}$ do not depend on $w$
and are estimated by Monte Carlo.

**Algorithm**, per gameweek:

1. $\mu_\delta$, $\Sigma_\delta$ from the predictions and their past errors
2. $\mathbb{W}_o$: the opponents' teams
3. Monte Carlo draws of $(\delta, G^{(r')})$
4. Solve for $w(\lambda)$ over a grid of $\lambda$ (mixed-integer quadratic program, SCIP)
5. $\lambda^*$: the $\lambda$ whose team beats $G^{(r')}$ in the most Monte Carlo draws

**Adapted to FantasyGo** (the original picks a 15-man FPL squad with transfers against one head-to-head opponent):

* $w$ is the 11 players with the captain counted 2x and the vice-captain 1.5x: 1 GK, 3-5 DEF, 2-5 MID, 1-3 FWD,
  a budget of 80, at most 3 players per club, unlimited changes each gameweek
* $\mathbb{W}_o$ is the contest field, either simulated (`FIELD = "simulated"`, the default) with the team generator
  of FPL-Optimal-Transfer's `team_modeling - team generator.ipynb` (Algorithm 1), fed by the selection model of
  `beta estimators.ipynb`, or every FantasyGo manager's actual team of the previous gameweek (`FIELD = "previous"`)
* $\mu_\delta$ is `pred` from `models/predictions.csv` (every player with a fixture). $\Sigma_\delta$ is the covariance of
  the errors against the actual points (`total_points - pred`), not xP as in the original: FantasyGo is scored on
  points, and this includes the risk of a player not playing
* players without enough error history get their position's median variance (the original gave them ~0)
* the decision is restricted to the `SHORTLIST` best-predicted players per position, and $\Sigma_\delta$ is factored
  ($\Sigma = LL^T$) so the quadratic term is a sum of squares, which SCIP solves far faster

In [1]:
from pathlib import Path

import joblib

import numpy as np
import pandas as pd
from pyscipopt import Model, quicksum

ROOT = Path("../..")

preds = pd.read_csv(ROOT / "models/predictions.csv")
preds["value"] = preds["value"] / 10
fg_picks = pd.read_csv(ROOT / "optimizer/data/fg_manager_picks.csv")
fg_managers = pd.read_csv(ROOT / "optimizer/data/fg_managers.csv")
alpha_by_gw = joblib.load("league_alpha_draws.pkl")  # from beta estimators.ipynb

### Parameters

In [ ]:
FIELD = "simulated"  # "simulated" (team generator) or "previous" (last gameweek's actual teams)
RESULTS = Path(f"mvt_results_{FIELD}.pkl")  # cached solutions: delete to recompute

# Team generator (Algorithm 1)
Q_STACK = 0.35  # probability that a simulated manager stacks teammates
BUDGET_LB = 0.8  # simulated teams spend at least this share of the budget (real teams: median 97%, 10% under 84%)
XP_ANCHOR_PCT = 75  # a stack's anchor must be predicted above this percentile

R_PCT = 0.8  # rank to beat in the field: 0.8 = top 20% of the teams
N_MC = 2000  # Monte Carlo draws
LAMBDA_GRID = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
SHORTLIST = 30  # candidates per position, by predicted points
TIME_LIMIT = 30  # seconds per solve
MIN_HISTORY = 3  # gameweeks of prediction errors needed for Sigma
SEED = 0

BUDGET = 80.0
XI_SIZE = 11
FORMATION = {"GK": (1, 1), "DEF": (3, 5), "MID": (2, 5), "FWD": (1, 3)}
MAX_PER_TEAM = 3
CAPTAIN, VICE = 2.0, 1.5

FIRST_GW = preds["round"].min() + MIN_HISTORY
print(
    f"MVT for gameweeks {FIRST_GW}-{preds['round'].max()}, beating the top {1 - R_PCT:.0%} of the field"
)

MVT for gameweeks 23-38, beating the top 20% of the field


### Step 1: $\mu_\delta$ and $\Sigma_\delta$

$\Sigma_\delta$ is the covariance of the prediction errors in the gameweeks before `gw` (pairwise, at least
`MIN_HISTORY` shared gameweeks), made positive semi-definite as in the original: clip the eigenvalues of the
correlation matrix, rescale it to a unit diagonal, and scale back by each player's own variance.

In [ ]:
def build_mu_sigma(gw, min_eig_frac=1e-4):
    players = preds[preds["round"] == gw].set_index("fpl_id")
    hist = preds[preds["round"] < gw]
    errors = (hist["total_points"] - hist["pred"]).rename("error")
    pivot = hist.assign(error=errors).pivot_table(
        index="round", columns="fpl_id", values="error"
    )
    cov = pivot.reindex(columns=players.index).cov(min_periods=MIN_HISTORY)

    var = pd.Series(np.diag(cov.to_numpy()), index=players.index)
    position_median = var[var > 0].groupby(players["pos"]).median()
    var = var.where(var > 0, players["pos"].map(position_median))  # not enough history

    std = np.sqrt(var.to_numpy())
    corr = cov.fillna(0.0).to_numpy() / np.outer(std, std)
    corr = np.clip((corr + corr.T) / 2, -1, 1)
    np.fill_diagonal(corr, 1.0)
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals = np.clip(eigvals, max(min_eig_frac * eigvals.max(), 1e-8), None)
    corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(corr))
    corr = corr / np.outer(d, d)
    sigma = corr * np.outer(std, std)
    return players, players["pred"].to_numpy(), (sigma + sigma.T) / 2

### Step 2: the field $\mathbb{W}_o$

Each team is a row of multipliers over the players (2 captain, 1.5 vice, 1 other). Players without a fixture in
`gw` score nothing and are left out.

**Simulated** (Algorithm 1 of the team generator), one team at a time:

1. draw one posterior draw of the selection model (its $\alpha$ for every candidate) and, per position, a selection
   vector $p \sim \text{Dirichlet}(\alpha)$
2. draw a formation from the FantasyGo managers' formations of the earlier gameweeks
3. sample the players of each position by $p$, without replacement
4. with probability `Q_STACK` the manager stacks: if the best selected forward is predicted above the
   `XP_ANCHOR_PCT` percentile, a midfield teammate (sampled by $p$) replaces the weakest selected midfielder; the same
   for the best defender and a goalkeeper of his team
5. keep the team if it costs between `BUDGET_LB` x budget and the budget and has at most 3 players per club;
   otherwise start again
6. captain and vice-captain: the two highest predictions (the rule of `selected_info`)

As many teams are simulated as there were entries in the previous gameweek. Unlike the original, duplicate teams
are kept: a real field has them too.

**Previous**: every FantasyGo manager's team of gameweek `gw - 1`, with their captains and vice-captains.

In [ ]:
FORMATIONS = [
    (d, m, 10 - d - m) for d in range(3, 6) for m in range(2, 6) if 1 <= 10 - d - m <= 3
]


def formation_frequencies(gw):
    """Share of each formation (DEF, MID, FWD) among the FantasyGo teams before gw."""
    earlier = fg_picks[fg_picks["gameweek"] < gw]
    counts = earlier.groupby(["row", "position"]).size().unstack(fill_value=0)
    shape = counts.reindex(columns=["DEF", "MID", "FWD"], fill_value=0).apply(
        tuple, axis=1
    )
    freq = (
        shape.value_counts().reindex(FORMATIONS, fill_value=0) + 1
    )  # every valid formation possible
    return freq / freq.sum()


def candidate_pools(gw, players):
    """Per position: the generator's candidates with a prediction this gameweek, and their alpha draws."""
    pools = {}
    for pos, d in alpha_by_gw[gw].items():
        keep = np.isin(d["fpl_id"], players.index)
        ids = d["fpl_id"][keep]
        pools[pos] = {
            "ids": ids,
            "alpha": d["alpha"][:, keep].astype(float),
            "value": players.loc[ids, "value"].to_numpy(),
            "team": players.loc[ids, "team"].to_numpy(),
            "pred": players.loc[ids, "pred"].to_numpy(),
        }
    return pools


def selection_vector(alpha, rng):
    """p ~ Dirichlet(alpha), robust to very small alphas (gamma draws that all underflow to 0)."""
    g = rng.gamma(alpha)
    p = g / g.sum() if g.sum() > 0 else alpha / alpha.sum()
    p = np.maximum(p, 1e-12)
    return p / p.sum()


def stack(selected, p, pool_from, pool_to, anchor_cut, rng, replace_worst=True):
    """Replace a selected player of pool_to with a teammate of the best selected player of pool_from."""
    best = selected["from"][np.argmax(pool_from["pred"][selected["from"]])]
    if pool_from["pred"][best] < anchor_cut:
        return selected["to"]
    mates = np.where(
        (pool_to["team"] == pool_from["team"][best])
        & ~np.isin(np.arange(len(p)), selected["to"])
    )[0]
    if len(mates) == 0:
        return selected["to"]
    mate = rng.choice(mates, p=p[mates] / p[mates].sum())
    out = selected["to"].copy()
    out[np.argmin(pool_to["pred"][out]) if replace_worst else 0] = mate
    return out


def simulate_team(pools, formations, anchor_cut, rng):
    """One team (fpl ids, multipliers), or None if it breaks the budget or club rules."""
    s = rng.integers(len(next(iter(pools.values()))["alpha"]))
    form = formations.index[rng.choice(len(formations), p=formations.to_numpy())]
    sizes = {"GK": 1, "DEF": form[0], "MID": form[1], "FWD": form[2]}
    p = {pos: selection_vector(pools[pos]["alpha"][s], rng) for pos in pools}
    chosen = {
        pos: rng.choice(len(p[pos]), size=n, replace=False, p=p[pos])
        for pos, n in sizes.items()
    }
    if rng.random() < Q_STACK:
        chosen["MID"] = stack(
            {"from": chosen["FWD"], "to": chosen["MID"]},
            p["MID"],
            pools["FWD"],
            pools["MID"],
            anchor_cut,
            rng,
        )
        chosen["GK"] = stack(
            {"from": chosen["DEF"], "to": chosen["GK"]},
            p["GK"],
            pools["DEF"],
            pools["GK"],
            anchor_cut,
            rng,
            replace_worst=False,
        )

    ids = np.concatenate([pools[pos]["ids"][chosen[pos]] for pos in sizes])
    cost = sum(pools[pos]["value"][chosen[pos]].sum() for pos in sizes)
    clubs = np.concatenate([pools[pos]["team"][chosen[pos]] for pos in sizes])
    if (
        not (BUDGET_LB * BUDGET <= cost <= BUDGET)
        or pd.Series(clubs).value_counts().max() > MAX_PER_TEAM
    ):
        return None
    pred = np.concatenate([pools[pos]["pred"][chosen[pos]] for pos in sizes])
    multiplier = np.ones(len(ids))
    order = np.argsort(-pred)
    multiplier[order[0]], multiplier[order[1]] = CAPTAIN, VICE
    return ids, multiplier


def simulate_field(gw, players, rng, n_teams=None, max_tries=200_000):
    """n_teams simulated teams (default: the previous gameweek's number of entries) and the acceptance rate."""
    n_teams = n_teams or fg_picks.loc[fg_picks["gameweek"] == gw - 1, "row"].nunique()
    pools = candidate_pools(gw, players)
    formations = formation_frequencies(gw)
    anchor_cut = np.percentile(players["pred"], XP_ANCHOR_PCT)
    teams, tries = [], 0
    while len(teams) < n_teams and tries < max_tries:
        tries += 1
        team = simulate_team(pools, formations, anchor_cut, rng)
        if team is not None:
            teams.append(team)
    return teams, len(teams) / tries


def previous_field(gw):
    """The FantasyGo teams of gameweek gw - 1 as (fpl ids, multipliers)."""
    return entries(gw - 1)


def entries(gw):
    """The FantasyGo teams of gameweek gw as (fpl ids, multipliers)."""
    picks = fg_picks[fg_picks["gameweek"] == gw]
    return [
        (
            g["player_id"].to_numpy(),
            np.where(g["is_captain"], CAPTAIN, np.where(g["is_vice"], VICE, 1.0)),
        )
        for _, g in picks.groupby("row")
    ]


def field_matrix(gw, players, rng):
    """W_o: one row of multipliers over `players` per team of the field."""
    teams = (
        simulate_field(gw, players, rng)[0]
        if FIELD == "simulated"
        else previous_field(gw)
    )
    position = {e: i for i, e in enumerate(players.index)}
    W_o = np.zeros((len(teams), len(players)))
    for k, (ids, multiplier) in enumerate(teams):
        for e, m in zip(ids, multiplier):
            if e in position:
                W_o[k, position[e]] += m
    return W_o

#### Is the simulated field like the real one?

For every gameweek, the simulated field next to the real contest of that gameweek and the previous gameweek's
teams (the other `FIELD`), all scored with the actual points:

* `cost`: mean team cost
* `share_corr`: rank correlation of how often each player is picked, against the real field
* `cut_points`: the points needed for the top `1 - R_PCT` of the field, the $G^{(r')}$ the optimizer has to beat

In [ ]:
def field_summary(teams, gw):
    actual = preds[preds["round"] == gw].set_index("fpl_id")
    points = [
        (actual["total_points"].reindex(ids).fillna(0).to_numpy() * m).sum()
        for ids, m in teams
    ]
    cost = [actual["value"].reindex(ids).sum() for ids, _ in teams]
    picks = pd.Series(np.concatenate([ids for ids, _ in teams])).value_counts()
    return {
        "cost": np.mean(cost),
        "cut_points": np.quantile(points, R_PCT),
        "mean_points": np.mean(points),
    }, picks


rows = []
for gw in range(FIRST_GW, preds["round"].max() + 1):
    rng = np.random.default_rng(SEED + gw)
    players = preds[preds["round"] == gw].set_index("fpl_id")
    real, real_picks = field_summary(entries(gw), gw)
    sim_teams, accepted = simulate_field(gw, players, rng)
    for name, teams in [("simulated", sim_teams), ("previous", previous_field(gw))]:
        summary, picks = field_summary(teams, gw)
        both = pd.concat([picks, real_picks], axis=1).fillna(0)
        rows.append(
            {
                "round": gw,
                "field": name,
                **summary,
                "share_corr": both.corr(method="spearman").iloc[0, 1],
                "accept_rate": accepted if name == "simulated" else np.nan,
            }
        )
    rows.append({"round": gw, "field": "real", **real})
fields = pd.DataFrame(rows)
fields.groupby("field")[
    ["cost", "cut_points", "mean_points", "share_corr", "accept_rate"]
].mean().round(2)

,cost,cut_points,mean_points,share_corr,accept_rate
field,,,,,
previous,70.42,61.46,49.84,0.21,NaN
real,75.15,68.53,56.56,NaN,NaN
simulated,71.14,63.62,50.92,0.57,0.58


In [6]:
fields.pivot(index="round", columns="field", values="cut_points").round(1)

field,previous,real,simulated
round,,,
23,48.0,61.0,74.4
24,62.5,72.0,60.5
25,76.0,63.5,70.2
26,55.0,50.0,69.3
27,76.5,88.5,73.0
28,57.0,51.2,43.0
29,14.0,32.5,34.0
30,62.3,71.0,59.5
31,59.0,66.0,65.5


### Step 3: Monte Carlo $(\delta, G^{(r')})$

$G^{(r')}$ is the score of the team at rank `R_PCT` of the field in each draw.

In [ ]:
def monte_carlo(mu, sigma, W_o, rng):
    delta = rng.multivariate_normal(mu, sigma, size=N_MC, method="eigh")
    k = min(
        max(int(np.ceil((1 - R_PCT) * len(W_o))), 1), len(W_o)
    )  # r' counted from the top
    G = np.sort(delta @ W_o.T, axis=1)[:, -k]
    cov_delta_G = ((delta - mu) * (G - G.mean())[:, None]).mean(axis=0)
    return delta, G, cov_delta_G

### Step 4: $w(\lambda)$, a mixed-integer quadratic program

Variables per candidate: picked, captain and vice-captain (binary), and its multiplier
$w_i = \text{picked} + \text{captain} + 0.5\,\text{vice}$. With $\Sigma = LL^T$ and $y = L^Tw$, the variance is
$\sum_k y_k^2$. SCIP takes a linear objective, so the mean-variance objective is the variable $z$ with
$z \leq w^T\mu - s\lambda(\sum_k y_k^2 - 2w^T\sigma_{\delta,G})$. For $s = +1$ ($\mu_{Y_w} \geq 0$) this is convex
and solves quickly; for $s = -1$ it is not, and the solve stops at `TIME_LIMIT` with its best team.

In [ ]:
def shortlist(players):
    best = (
        players.reset_index()
        .groupby("pos", group_keys=False)
        .apply(lambda g: g.nlargest(SHORTLIST, "pred"))
    )
    return best.index.to_numpy()  # positions in `players`


def solve_for_lambda(cands, mu, sigma, cov_delta_G, lam, sense=1):
    n = len(cands)
    m = Model()
    m.hideOutput()
    m.setParam("limits/time", TIME_LIMIT)
    pick = [m.addVar(vtype="B") for _ in range(n)]
    captain = [m.addVar(vtype="B") for _ in range(n)]
    vice = [m.addVar(vtype="B") for _ in range(n)]
    w = [m.addVar(lb=0, ub=CAPTAIN) for _ in range(n)]
    for i in range(n):
        m.addCons(w[i] == pick[i] + (CAPTAIN - 1) * captain[i] + (VICE - 1) * vice[i])
        m.addCons(captain[i] + vice[i] <= pick[i])
    m.addCons(quicksum(pick) == XI_SIZE)
    m.addCons(quicksum(captain) == 1)
    m.addCons(quicksum(vice) == 1)
    m.addCons(quicksum(cands["value"].iat[i] * pick[i] for i in range(n)) <= BUDGET)
    for pos, (low, high) in FORMATION.items():
        n_pos = quicksum(pick[i] for i in np.where(cands["pos"].to_numpy() == pos)[0])
        m.addCons(n_pos >= low)
        m.addCons(n_pos <= high)
    for team in cands["team"].unique():
        m.addCons(
            quicksum(pick[i] for i in np.where(cands["team"].to_numpy() == team)[0])
            <= MAX_PER_TEAM
        )

    eigvals, eigvecs = np.linalg.eigh(sigma)
    keep = eigvals > 1e-9 * eigvals.max()
    L = eigvecs[:, keep] * np.sqrt(eigvals[keep])
    y = [m.addVar(lb=-1e4, ub=1e4) for _ in range(L.shape[1])]
    for k in range(L.shape[1]):
        m.addCons(y[k] == quicksum(L[i, k] * w[i] for i in range(n)))
    mean = quicksum(mu[i] * w[i] for i in range(n))
    variance = quicksum(v * v for v in y) - 2 * quicksum(
        cov_delta_G[i] * w[i] for i in range(n)
    )
    z = m.addVar(lb=-1e6, ub=1e6)
    m.addCons(z <= mean - sense * lam * variance)
    m.setObjective(z, "maximize")
    m.optimize()
    if m.getNSols() == 0:
        return None, m.getStatus()
    sol = m.getBestSol()
    team = cands.assign(
        w=[sol[v] for v in w],
        captain=[sol[v] > 0.5 for v in captain],
        vice=[sol[v] > 0.5 for v in vice],
    )
    return team[[sol[v] > 0.5 for v in pick]], m.getStatus()

### Step 5: $\lambda^*$, and the whole gameweek as a function

$\widehat{\text{Prob}}(Y_w > 0)$ is the share of the Monte Carlo draws where the team beats $G^{(r')}$. The sign
$s$ comes from the best-mean team ($\lambda = 0$).

In [ ]:
def run_mvt(gw, verbose=True):
    rng = np.random.default_rng(SEED + gw)
    players, mu, sigma = build_mu_sigma(gw)
    W_o = field_matrix(gw, players, rng)
    delta, G, cov_delta_G = monte_carlo(mu, sigma, W_o, rng)

    idx = shortlist(players)
    cands = players.reset_index().iloc[idx].reset_index(drop=True)
    mu_c, sigma_c, cov_c = mu[idx], sigma[np.ix_(idx, idx)], cov_delta_G[idx]

    frontier, teams, sense = [], {}, 1
    for lam in LAMBDA_GRID:
        team, status = solve_for_lambda(cands, mu_c, sigma_c, cov_c, lam, sense)
        if team is None:
            continue
        w = np.zeros(len(cands))
        w[team.index] = team["w"]
        if lam == 0:
            sense = 1 if mu_c @ w - G.mean() >= 0 else -1
        Y = delta[:, idx] @ w - G
        frontier.append(
            {
                "lambda": lam,
                "status": status,
                "prob_beat": (Y > 0).mean(),
                "mean": mu_c @ w,
                "sd": np.sqrt(w @ sigma_c @ w),
            }
        )
        teams[lam] = team
    frontier = pd.DataFrame(frontier)
    best = frontier.loc[frontier["prob_beat"].idxmax(), "lambda"]
    if verbose:
        print(
            f"GW{gw}: field of {len(W_o)}, G mean {G.mean():.1f} sd {G.std():.1f}, "
            f"lambda* = {best}, Prob(beat) {frontier['prob_beat'].max():.1%}"
        )
    return {
        "gw": gw,
        "frontier": frontier,
        "teams": teams,
        "lambda_star": best,
        "G_mean": G.mean(),
        "G_sd": G.std(),
    }

### One gameweek

In [10]:
example = run_mvt(30)
example["frontier"].round(3)

GW30: field of 228, G mean 64.6 sd 11.5, lambda* = 0.1, Prob(beat) 93.0%


,lambda,status,prob_beat,mean,sd
0,0.000,optimal,0.871,79.172,21.122
1,0.005,optimal,0.871,79.172,21.122
2,0.010,optimal,0.894,78.911,18.835
3,0.020,optimal,0.894,78.911,18.835
4,0.050,optimal,0.906,77.899,17.283
5,0.100,optimal,0.930,75.397,13.679
6,0.200,timelimit,0.886,71.763,11.852
7,0.500,timelimit,0.854,69.975,12.167


In [ ]:
example["teams"][example["lambda_star"]][
    ["pos", "name", "team", "value", "pred", "captain", "vice"]
]

,pos,name,team,value,pred,captain,vice
1,DEF,Cristian Romero,Spurs,5.1,5.827739,False,False
2,DEF,Malo Gusto,Chelsea,4.2,4.856106,False,False
3,DEF,Conor Bradley,Liverpool,4.2,4.594636,False,False
30,FWD,Rodrigo Muniz Carvalho,Fulham,4.6,7.243659,True,False
31,FWD,Ollie Watkins,Aston Villa,9.0,6.008747,False,False
61,GK,André Onana,Man Utd,4.8,4.443685,False,False
90,MID,Son Heung-min,Spurs,10.1,7.276041,False,True
91,MID,Mohamed Salah,Liverpool,13.2,6.788609,False,False
92,MID,Cole Palmer,Chelsea,5.8,6.732080,False,False
95,MID,James Maddison,Spurs,8.0,5.611114,False,False


### Every gameweek

Solutions are cached in `mvt_results.pkl` (delete it, or change a parameter and delete it, to recompute).

In [12]:
import pickle

if RESULTS.exists():
    results = pickle.loads(RESULTS.read_bytes())
else:
    results = {gw: run_mvt(gw) for gw in range(FIRST_GW, preds["round"].max() + 1)}
    RESULTS.write_bytes(pickle.dumps(results))

GW23: field of 284, G mean 65.8 sd 7.5, lambda* = 0.2, Prob(beat) 92.5%


GW24: field of 261, G mean 72.5 sd 7.7, lambda* = 0.05, Prob(beat) 92.8%


GW25: field of 269, G mean 75.8 sd 8.3, lambda* = 0.2, Prob(beat) 93.2%


GW26: field of 317, G mean 77.9 sd 14.7, lambda* = 0.5, Prob(beat) 98.9%


GW27: field of 247, G mean 71.5 sd 10.5, lambda* = 0.5, Prob(beat) 99.6%


GW28: field of 226, G mean 71.5 sd 12.4, lambda* = 0.1, Prob(beat) 96.7%


... (6 outputs left out)

GW35: field of 203, G mean 75.7 sd 7.8, lambda* = 0.1, Prob(beat) 98.4%


GW36: field of 207, G mean 79.3 sd 9.9, lambda* = 0.05, Prob(beat) 96.7%


GW37: field of 229, G mean 79.6 sd 7.0, lambda* = 0.1, Prob(beat) 98.4%


GW38: field of 243, G mean 79.3 sd 8.6, lambda* = 0.2, Prob(beat) 94.0%


### How did the teams actually do?

Each team scored on the actual points (captain x2, vice x1.5) and ranked in the gameweek's FantasyGo contest:
rank = 1 + entries with more points; `prize` is what the entry at that rank won. The mean-variance team
($\lambda^*$) is compared with the best-mean team ($\lambda = 0$, the base optimizer on the same shortlist).

In [ ]:
def team_points(team, gw):
    actual = preds[preds["round"] == gw].set_index("fpl_id")["total_points"]
    return float((team["fpl_id"].map(actual).fillna(0) * team["w"]).sum())


def contest_rank(points, gw):
    field = fg_managers[fg_managers["gameweek"] == gw].sort_values(
        "points", ascending=False
    )
    rank = int((field["points"] > points).sum()) + 1
    prize = field["prize"].iloc[rank - 1] if rank <= len(field) else 0.0
    return rank, len(field), prize


rows = []
for gw, res in results.items():
    row = {"round": gw, "lambda_star": res["lambda_star"], "G_mean": res["G_mean"]}
    for label, lam in [("mvt", res["lambda_star"]), ("base", 0.0)]:
        team = res["teams"][lam]
        f = res["frontier"].set_index("lambda").loc[lam]
        points = team_points(team, gw)
        rank, entries, prize = contest_rank(points, gw)
        row |= {
            f"{label}_prob": f["prob_beat"],
            f"{label}_mean": f["mean"],
            f"{label}_sd": f["sd"],
            f"{label}_points": points,
            f"{label}_rank": rank,
            f"{label}_prize": prize,
        }
    row["entries"] = entries
    row["target_rank"] = int(np.ceil((1 - R_PCT) * entries))
    rows.append(row)
evaluation = pd.DataFrame(rows).set_index("round").round(2)

for label in ["mvt", "base"]:
    hit = (evaluation[f"{label}_rank"] <= evaluation["target_rank"]).sum()
    print(
        f"{label:4s}: {evaluation[f'{label}_points'].sum():.0f} points, median rank top "
        f"{(evaluation[f'{label}_rank'] / evaluation['entries']).median():.0%}, in the top {1 - R_PCT:.0%} in "
        f"{hit} of {len(evaluation)} gameweeks (predicted {evaluation[f'{label}_prob'].mean():.0%}), "
        f"prizes {evaluation[f'{label}_prize'].sum():,.2f}"
    )
evaluation

mvt : 937 points, median rank top 50%, in the top 20% in 3 of 16 gameweeks (predicted 95%), prizes 58.35
base: 1034 points, median rank top 42%, in the top 20% in 2 of 16 gameweeks (predicted 83%), prizes 1,278.81


,lambda_star,G_mean,mvt_prob,mvt_mean,mvt_sd,mvt_points,mvt_rank,mvt_prize,base_prob,base_mean,base_sd,base_points,base_rank,base_prize,entries,target_rank
round,,,,,,,,,,,,,,,,
23,0.20,65.82,0.92,75.20,6.58,65.0,9,25.00,0.70,77.21,21.23,41.0,102,0.00,261,53
24,0.05,72.50,0.93,85.11,7.22,36.5,130,0.00,0.71,86.69,26.00,35.0,142,0.00,269,54
25,0.20,75.79,0.93,83.92,7.20,61.0,223,0.00,0.80,87.08,17.30,71.5,158,0.00,317,64
26,0.50,77.86,0.99,89.19,14.40,77.5,7,10.00,0.82,93.01,29.49,83.5,2,1204.85,247,50
27,0.50,71.55,1.00,83.65,10.26,59.5,204,0.00,0.91,90.13,22.60,87.0,54,23.35,226,46
28,0.10,71.45,0.97,82.09,13.34,28.0,204,0.00,0.80,87.06,28.32,38.0,199,0.00,204,41
29,0.20,69.84,0.87,73.41,15.66,36.0,21,23.35,0.84,75.59,15.23,25.0,100,0.00,228,46
30,0.10,64.58,0.93,75.40,13.68,65.0,96,0.00,0.87,79.17,21.12,69.0,62,0.00,241,49
31,0.05,64.76,0.96,76.70,11.74,55.0,109,0.00,0.85,78.56,17.82,65.0,50,23.85,217,44


### Simulated field against the previous gameweek's teams

Both runs side by side, when both caches exist (run the notebook once with each `FIELD`).

In [ ]:
def summarize(path):
    res = pickle.loads(Path(path).read_bytes())
    out = []
    for gw, r in res.items():
        team = r["teams"][r["lambda_star"]]
        points = team_points(team, gw)
        rank, entries, prize = contest_rank(points, gw)
        out.append(
            {
                "round": gw,
                "lambda_star": r["lambda_star"],
                "G_mean": r["G_mean"],
                "points": points,
                "rank": rank,
                "top": rank <= np.ceil((1 - R_PCT) * entries),
                "prize": prize,
            }
        )
    return pd.DataFrame(out).set_index("round")


runs = {
    f: summarize(f"mvt_results_{f}.pkl")
    for f in ["simulated", "previous"]
    if Path(f"mvt_results_{f}.pkl").exists()
}
pd.DataFrame(
    {
        f: {
            "points": r["points"].sum(),
            "median_rank": r["rank"].median(),
            "top_20pct_weeks": r["top"].sum(),
            "prizes": r["prize"].sum(),
            "mean_G": r["G_mean"].mean(),
        }
        for f, r in runs.items()
    }
).round(1)

,simulated,previous
points,937.0,1006.5
median_rank,119.5,123.0
top_20pct_weeks,3.0,5.0
prizes,58.4,1344.0
mean_G,73.0,67.1


### Caveats

* $\mu_\delta$ over-predicts the top players (see `optimizer.ipynb`), so $\widehat{\text{Prob}}$ is optimistic: both
  teams and the field are scored with the same biased $\mu_\delta$, but our team is chosen for high $\mu_\delta$.
* $\Sigma_\delta$ rests on 3 to 18 gameweeks of errors, so its correlations are noisy; the eigenvalue floor keeps
  it usable but does not make it accurate.
* The simulated field rests on the selection model and on the captain rule (two highest predictions); the
  previous-gameweek field ignores that managers can change all 11 players each gameweek.
* Only the `SHORTLIST` best-predicted players per position are candidates.